In [ ]:
# Orquestrador Universal Controlado por YAML (Contract-Driven ML)
import yaml
import pandas as pd
import boto3
import io
import os
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# 1. Carregar Contrato Padrão do S3 (MinIO)
s3_client = boto3.client('s3', endpoint_url='http://minio:9000', aws_access_key_id='minioadmin', aws_secret_access_key='minioadmin', region_name='us-east-1')

# Puxamos o contrato recém enviado para o Bucket model-contracts
response_yaml = s3_client.get_object(Bucket='model-contracts', Key='penguins_contract.yaml')
contract = yaml.safe_load(response_yaml['Body'].read())

print(f"Lendo yaml de Especificação via nuvem: {contract['metadata']['name']} (v{contract['metadata']['version']})")

os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio:9000"

mlflow.set_tracking_uri("http://mlflow-server:5000")
mlflow.set_experiment(contract['metadata']['name'])


In [ ]:
# 2. Extração via Source (Mapeamento do YAML)
s3_client = boto3.client('s3', endpoint_url='http://minio:9000', aws_access_key_id='minioadmin', aws_secret_access_key='minioadmin', region_name='us-east-1')
response = s3_client.get_object(Bucket=contract['data']['bucket'], Key=contract['data']['file_name'])
df = pd.read_csv(io.BytesIO(response['Body'].read())).dropna()
print(df.head(2))

In [ ]:
# 3. Dynamic Pipeline Building (Montagem baseado estritamente no contrato)
target_col = contract['label']['column']
y = df[target_col]
X = df.drop(target_col, axis=1)

numeric_features = []
categorical_features = []

for feat in contract['features']:
    col_name = feat['name']
    if feat['dtype'] == 'float' or feat['dtype'] == 'int':
        numeric_features.append(col_name)
    elif feat['dtype'] == 'categorical':
        categorical_features.append(col_name)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
# 4. Instanciação Automática do Modelo Vencedor
alg = contract['model']['algorithm']
params = contract['model']['hyperparameters']

if alg == 'decision_tree':
    clf = DecisionTreeClassifier(**params)
elif alg == 'random_forest':
    clf = RandomForestClassifier(**params)
elif alg == 'logistic_regression':
    clf = LogisticRegression(**params)
else:
    raise ValueError("Algoritmo não suportado")

# Encapsula ABSOLUTAMENTE TUDO em um objeto Sklearn Pipeline Genérico
pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])

# 5. Particionamento e Rastreabilidade MLflow
test_size = contract['data']['splits']['test']
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=test_size, random_state=42)

with mlflow.start_run(run_name=f"Run_YAML_Auto_{contract['metadata']['version']}"):
    
    # SALVANDO O PRÓPRIO ARQUIVO DE CONTRATO (DOCUMENTAÇÃO DO PROJETO VAI JUNTO PRO S3)
    mlflow.log_dict(contract, "contract_snapshot.yaml")
    
    # O pipeline compila a imputação, scalonamento, e fit de uma só vez
    pipeline.fit(X_treino, y_treino)
    previsoes = pipeline.predict(X_teste)
    
    if contract['problem']['type'] == 'multiclass_classification':
        f1 = f1_score(y_teste, previsoes, average='weighted')
    else:
        f1 = f1_score(y_teste, previsoes)
        
    mlflow.log_metric(contract['problem']['target_metric'], f1)
    
    mlflow.log_param("algoritmo_engine", alg)
    mlflow.log_params(params)
    
    # Envio do artefato unificado
    mlflow.sklearn.log_model(pipeline, "pipeline_scikit_learn_completo")
    print(f"Pipeline Universal finalizado e empacotado! {contract['problem']['target_metric']}: {f1:.4f}")
    print("O Pipeline Sklearn inteiro (pre-processadores matemáticos + algoritmo) foi salvo no S3.")